# Diet Problem with AMPL in Python
## AMPL Modeling for Book Problem 2.10

This notebook solves the diet problem from book section `2.10` with AMPL from Python using `amplpy`.

We solve:
- the base mixed model from the book
- the student variation with explicit upper nutritional limits using the classroom assumption `max_requirement = 1.8 * min_requirement`


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```


In [1]:
from __future__ import annotations

from functools import wraps
from math import isclose
from time import perf_counter


In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


def create_ampl_instance(solver: str = "highs"):
    return ampl_notebook(modules=[solver], license_uuid="default")


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
CONTINUOUS_FOODS = ["milk", "beans", "rice", "noodles", "butter"]
INTEGER_FOODS = ["eggs", "carrots", "apples", "oranges"]
NUTRIENTS = ["protein", "fat", "carbs", "vitamin_a", "vitamin_c", "folate", "calcium", "iron"]

COST = {
    "milk": 660,
    "eggs": 1180,
    "beans": 2100,
    "rice": 950,
    "noodles": 560,
    "carrots": 440,
    "apples": 1440,
    "oranges": 400,
    "butter": 800,
}

MIN_REQUIREMENT = {
    "protein": 65,
    "fat": 66,
    "carbs": 362,
    "vitamin_a": 700,
    "vitamin_c": 75,
    "folate": 400,
    "calcium": 1000,
    "iron": 18,
}

FOOD_NUTRIENTS = {
    ("milk", "protein"): 3.3, ("eggs", "protein"): 13.5, ("beans", "protein"): 20.6, ("rice", "protein"): 6.4, ("noodles", "protein"): 12.2, ("carrots", "protein"): 0.9, ("apples", "protein"): 0.3, ("oranges", "protein"): 0.7, ("butter", "protein"): 0,
    ("milk", "fat"): 3.2, ("eggs", "fat"): 10, ("beans", "fat"): 1.6, ("rice", "fat"): 0.8, ("noodles", "fat"): 0.3, ("carrots", "fat"): 0.5, ("apples", "fat"): 0.3, ("oranges", "fat"): 0.3, ("butter", "fat"): 82.9,
    ("milk", "carbs"): 4.8, ("eggs", "carbs"): 4, ("beans", "carbs"): 57.3, ("rice", "carbs"): 79.7, ("noodles", "carbs"): 74.6, ("carrots", "carbs"): 8.1, ("apples", "carbs"): 14.5, ("oranges", "carbs"): 8.7, ("butter", "carbs"): 0,
    ("milk", "vitamin_a"): 91, ("eggs", "vitamin_a"): 226, ("beans", "vitamin_a"): 85, ("rice", "vitamin_a"): 0, ("noodles", "vitamin_a"): 0, ("carrots", "vitamin_a"): 1346, ("apples", "vitamin_a"): 4, ("oranges", "vitamin_a"): 40, ("butter", "vitamin_a"): 5.6,
    ("milk", "vitamin_c"): 4.5, ("eggs", "vitamin_c"): 0, ("beans", "vitamin_c"): 26, ("rice", "vitamin_c"): 0, ("noodles", "vitamin_c"): 0, ("carrots", "vitamin_c"): 6, ("apples", "vitamin_c"): 10, ("oranges", "vitamin_c"): 50, ("butter", "vitamin_c"): 0,
    ("milk", "folate"): 14, ("eggs", "folate"): 51.2, ("beans", "folate"): 88, ("rice", "folate"): 12, ("noodles", "folate"): 4, ("carrots", "folate"): 10, ("apples", "folate"): 5, ("oranges", "folate"): 37, ("butter", "folate"): 3,
    ("milk", "calcium"): 123, ("eggs", "calcium"): 100, ("beans", "calcium"): 260, ("rice", "calcium"): 14, ("noodles", "calcium"): 24, ("carrots", "calcium"): 41, ("apples", "calcium"): 6, ("oranges", "calcium"): 36, ("butter", "calcium"): 0,
    ("milk", "iron"): 0.1, ("eggs", "iron"): 2.4, ("beans", "iron"): 2, ("rice", "iron"): 0.8, ("noodles", "iron"): 3.6, ("carrots", "iron"): 0.7, ("apples", "iron"): 0.4, ("oranges", "iron"): 0.3, ("butter", "iron"): 0,
}


In [4]:
@timed
def solve_diet_with_ampl(max_factor=None, solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set C ordered;
        set D ordered;
        set N ordered;

        param cost {C union D};
        param min_req {N};
        param nutr {C union D, N};
        param use_max_bounds binary;
        param max_req {N} default 0;

        var x {c in C} >= 0;
        var y {d in D} integer >= 0;

        minimize TotalCost:
            sum {c in C} cost[c] * x[c] + sum {d in D} cost[d] * y[d];

        subject to MinimumNutrition {n in N}:
            sum {c in C} nutr[c, n] * x[c] + sum {d in D} nutr[d, n] * y[d] >= min_req[n];

        subject to MaximumNutrition {n in N: use_max_bounds = 1}:
            sum {c in C} nutr[c, n] * x[c] + sum {d in D} nutr[d, n] * y[d] <= max_req[n];
        '''
    )
    ampl.set["C"] = CONTINUOUS_FOODS
    ampl.set["D"] = INTEGER_FOODS
    ampl.set["N"] = NUTRIENTS
    ampl.param["cost"] = COST
    ampl.param["min_req"] = MIN_REQUIREMENT
    ampl.param["nutr"] = FOOD_NUTRIENTS
    ampl.param["use_max_bounds"] = 0 if max_factor is None else 1
    if max_factor is not None:
        ampl.param["max_req"] = {nutrient: max_factor * MIN_REQUIREMENT[nutrient] for nutrient in NUTRIENTS}
    ampl.solve(solver=solver)

    x_values = ampl.get_variable("x").get_values().to_dict()
    y_values = ampl.get_variable("y").get_values().to_dict()
    result = {
        food: round(x_values.get(food, x_values.get((food,), 0.0)), 3)
        for food in CONTINUOUS_FOODS
    }
    result.update(
        {
            food: int(round(y_values.get(food, y_values.get((food,), 0.0))))
            for food in INTEGER_FOODS
        }
    )
    result["cost"] = round(ampl.get_objective("TotalCost").value(), 3)
    return result


In [5]:
base_result, base_time = solve_diet_with_ampl()
print("=== BASE DIET RESULTS WITH AMPL ===")
print(f"Solution -> {base_result}")
print(f"Time     -> {base_time:.8f} seconds")
print("Note     -> this exact optimum is cheaper than the table reported in the book.")

assert isclose(base_result["cost"], 9425.771, abs_tol=1e-3)


/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 9425.771151
7 simplex iterations
1 branching nodes
=== BASE DIET RESULTS WITH AMPL ===
Solution -> {'milk': 4.415, 'beans': 0.275, 'rice': 0, 'noodles': 4.058, 'butter': 0.577, 'eggs': 0, 'carrots': 0, 'apples': 0, 'oranges': 8, 'cost': 9425.771}
Time     -> 6.63506513 seconds
Note     -> this exact optimum is cheaper than the table reported in the book.


In [6]:
variant_result, variant_time = solve_diet_with_ampl(max_factor=1.8)
print("=== UPPER-BOUND VARIATION RESULTS WITH AMPL ===")
print(f"Solution -> {variant_result}")
print(f"Time     -> {variant_time:.8f} seconds")

assert isclose(variant_result["cost"], 10473.398, abs_tol=1e-3)


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 10473.39792
9 simplex iterations
1 branching nodes
=== UPPER-BOUND VARIATION RESULTS WITH AMPL ===
Solution -> {'milk': 0, 'beans': 2.833, 'rice': 0, 'noodles': 2.452, 'butter': 0.488, 'eggs': 2, 'carrots': 0, 'apples': 0, 'oranges': 1, 'cost': 10473.398}
Time     -> 3.54617596 seconds
